In [1]:
import numpy as np
from numpy import testing as npt

# Data Representation and Disk IO: Performance Beyond RAM

In this session, we extend our understanding of performance beyond CPU and memory. Many scientific workflows become slow not because of computation, but because of how we store and retrieve data. Here we examine:

  - How file formats affect size and speed
  - How precision choices influence storage
  - How disk activity differs from memory activity

We will measure what happens when we write arrays to disk, compare text and binary formats, and explore how data types determine both memory usage and file size. Throughout, we focus on making informed design decisions rather than relying on defaults.

## Utility Functions

In [15]:
import os
import psutil

def _format_bytes(bytes: float, precision: int = 2) -> str:
    """
    Takes a time in seconds and returns a string (e.g. ) that is more human-readable.

    Looking to do this in a real project?  Some alternatives:
      - `humanfriendly`: https://pypi.org/project/humanfriendly/#getting-started
    """

    if bytes < 0:
        raise ValueError("bytes must be non-negative")

    units = [("KB", 1000), ("MB", 1_000_000), ("GB", 1_000_000_000), ("TB", 1_000_000_000_000)]

    for unit, scale in reversed(units):
        if bytes >= scale:
            value = bytes / scale
            return f"{value:.{precision}f} {unit}"
    else:
        return f"{bytes} B"
    

def _disk_read() -> float:
    return psutil.disk_io_counters().read_bytes

def _disk_write() -> float:
    return psutil.disk_io_counters().write_bytes


class utils:
    format_bytes = _format_bytes
    bytes_read = _disk_read
    bytes_written = _disk_write

---

## Section 1: Reading Should Be Simple: Text vs Binary File Formats

When we save arrays to disk, we choose a representation. That choice affects:

  - File size 
  - Write speed
  - Read speed
  - CPU usage during IO (Reading and Writing)

In this section, we compare binary storage (.npy) with text-based storage (.txt). We measure disk writes directly and observe how closely file size reflects the array’s size in RAM. Our goal is to understand what is predictable, what is expensive, and why.

We will:

  - Measure memory usage with .nbytes
  - Measure disk writes using psutil
  - Compare CPU time and wall time
  - Examine how formatting affects text output


  ### Reference

  | Code                    | Description                                                |
| ----------------------- | ---------------------------------------------------------- |
| `np.arange()`           | Create arrays of controlled size and dtype for experiments |
| `array.nbytes`          | Number of bytes used by the array in RAM                   |
| `np.save()`             | Save array to binary `.npy` format                         |
| `np.savetxt()`          | Save array to text file                                    |
| `np.savetxt(fmt=...)`   | Control formatting of values when writing text             |
| `%time`                 | Measure CPU time and wall time in notebooks                |
| `utils.bytes_written()` | Measure bytes written to disk by this process              |
| `utils.format_bytes()`  | Convert byte counts into human-readable units              |


### Exercises

#### Writing to Binary NPY files with `np.save()`

the `np.save()` function is very simple: it puts two things into an `.npy` file:
  1. Writes a text header explaining the size, shape, and dtype of the array stored.
  2. Writes the array data in binary (a.k.a. "bytes") format.

That should make `np.save()` predictable and straightforward.  Let's try it out!

**Example**: Make an array of 200,000 float64 values and save the it to disk with `np.save()`, and print: 
1. How many bytes does the array take up in RAM?
2. How many bytes did the disk write when writing the file?

Is the amount of data written to these binary files about the same amount as the array stored in RAM? 

In [5]:
data.dtype

dtype('float64')

In [7]:
64 * 200_000 / 8

1600000.0

In [8]:
data.nbytes

1600000

In [9]:
data = np.arange(200_000, dtype=np.float64)
print('Size in RAM:', utils.format_bytes(data.nbytes))

dr0 = utils.bytes_written()
np.save('data.npy', data)
dr1 = utils.bytes_written()
print('Bytes Written:', utils.format_bytes(dr1 - dr0))

Size in RAM: 1.60 MB
Bytes Written: 1.60 MB


**Exercise**: Make an array of 1,000,000 float64 values and save the it to disk with `np.save()`, and print: 
1. How many bytes does the array take up in RAM?
2. How many bytes did the disk write when writing the file?

Is the amount of data written to these binary files about the same amount as the array stored in RAM? 

In [22]:
data = np.arange(200_000, dtype=np.uint16)
print('Size in RAM:', utils.format_bytes(data.nbytes))

dr0 = utils.bytes_written()
np.save('data.npy', data)
dr1 = utils.bytes_written()
print('Bytes Written:', utils.format_bytes(dr1 - dr0))

Size in RAM: 400.00 KB
Bytes Written: 0 B


**Exercise**: Does data type affect the data sizes, and/or the relative sizes between RAM and Disk when saving binary data with `np.save()`?  Let's try it again, this time with 1,000,000 **int64** values:

**Exercise**: How Much time does it take to write these files? Let's do a basic measurement: 
  - Create a data array that takes up 500 MB *when written to disk*.
  - Use `%time` to measure how much CPU and Wall time the writing took.  

How long did the data take to write?  Did the CPU have to do much work during that process?

#### Writing to Text files with `np.savetxt()`

the `np.savetxt()` function is also very simple; for each value in an array, it writes out the value in text and puts a seperator character (often a new-line) between each value.  

This is the same way that we record values on to paper, and makes reading these files in text editors to browse the data quite convenient.

That should make `np.savetxt()` also predictable and straightforward.  Let's try it out!

**Exercise**: Make an array of 1,000,000 float64 values and save the it to disk with `np.savetxt()`, and print: 
1. How many bytes does the array take up in RAM?
2. How many bytes did the disk write when writing the file?

Is the amount of data written to text files about the same amount as the array stored in RAM? 

**Exercise**: Does data type affect the data sizes, and/or the relative sizes between RAM and Disk when saving binary data with `np.savetxt(fmt='%d')`?  Let's try it again, this time with 1,000,000 **int64** values:

*Note*: the `fmt=` option changes how the text is formatted when the data is written.  `'%d'` means "write as integers".  `"%.18e"` is the default; it means "write in scientific floating point notation with 18 significance points"

**Exercise**: How Much time does it take to write these files? Let's do a basic measurement: 
  - Create a data array that takes up 300 MB *when written to disk*.
  - Use `%time` to measure how much CPU and Wall time the writing took.  

How long did the data take to write?  Did the CPU have to do much work during that process?

---

## Section 2: Precision as a Design Decision

Data type is not an implementation detail — it is a design choice.

The number of bits we allocate determines:

  - Range of representable values
  - Precision of numerical values
  - Memory usage
  - Disk size
  - Compression behaviour

In many scientific workflows, default data types (such as float64) are used automatically. However, this may be unnecessary and costly. In this section, we explore integer and floating-point limits and learn how to safely reduce storage size without compromising data integrity.

We will:

  - Inspect value ranges of integer types
  - Examine floating-point precision limits
  - Use astype() to reduce storage size
  - Verify correctness using NumPy testing utilities

### Exercises

#### Values, Precision, and Memory Size of Data Types

**Exercise**: How much space do these data types take up, in bytes?  Use `np.dtype().itemsize` to print the byte size of:
  - 16-bit floats (`np.float16`)
  - 32-bit floats (`np.float32`)
  - 64-bit floats (`np.float64`)
  
How many bytes do they take up?  Are they different?

Code snippet: `np.dtype(np.float16).itemsize`

In [25]:
np.dtype(np.float16).itemsize

2

**Exercise**: How much space do these data types take up, in bytes?  Use `np.dtype().itemsize` to print the byte size of:
  - 16-bit floats (`np.float16`)
  - 16-bit ints (`np.int16`)
  - 16-bit unsigned ints (`np.uint16`)
  
How many bytes do they take up?  Are they different?


**Exercise**: Boolean values are sometimes surprising in how they are stored in memory by Numpy; because they only store `True` and `False` values, by rights they should only take up 1 bit (1/8 of a byte).  Let's check if that's true.  How much space do these data types take up, in bytes?  Use `np.dtype().itemsize` to print the byte size of:
  - bools (`np.bool`)
  - 8-bit ints (`np.int8`)
  
How many bytes do they take up?  Are they different from each other?


**Exercise**: What values can `unsigned integers` hold?  Is there a big difference between, say, an 8-bit integer and a 32-bit integer?  Print the output of `np.iinfo()` and Compare `np.uint8`, `np.uint16`, `np.uint32`, and `np.uint64` and examine the minimum and maximum values for those data types.

*Code Snippet*: `print(np.iinfo(np.uint8))`

In [32]:
print(np.iinfo(np.uint8))

Machine parameters for uint8
---------------------------------------------------------------
min = 0
max = 255
---------------------------------------------------------------



**Exercise**: What values can `signed integers` hold?  Print the output of `np.iinfo()` and Compare `np.int8`, `np.int16`, `np.int32`, and `np.int64`.

What makes the values different than their corresponeding unsigned values? 

**Exercise**: What values can `floats` hold, and what is an attribute that makes them different from ints?  Print the output of `np.finfo()` and Compare `np.float16`, `np.float32`, and `np.float64`:

In [35]:
print(np.finfo(np.float64))

Machine parameters for float64
---------------------------------------------------------------
precision =  15   resolution = 1.0000000000000001e-15
machep =    -52   eps =        2.2204460492503131e-16
negep =     -53   epsneg =     1.1102230246251565e-16
minexp =  -1022   tiny =       2.2250738585072014e-308
maxexp =   1024   max =        1.7976931348623157e+308
nexp =       11   min =        -max
smallest_normal = 2.2250738585072014e-308   smallest_subnormal = 4.9406564584124654e-324
---------------------------------------------------------------



#### Reducing Data Size with `np.astype()`

**Example**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `np.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [229]:
temperature_c = np.random.randint(low=-15, high=35, size=10_000)
print(utils.format_bytes(temperature_c.nbytes))

temperature_c2 = np.astype(temperature_c, np.int8)
print(utils.format_bytes(temperature_c2.nbytes))

npt.assert_equal(temperature_c, temperature_c2)


40.00 KB
10.00 KB


**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `np.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [222]:
pixel_brightnesses = np.random.randint(low=0, high=256, size=10_000)


array([13.74993649,  2.53590496,  1.62495635, ...,  1.02192348,
        1.65743352, 10.79887842], shape=(10000,))

**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `np.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [36]:
binned_spike_counts = np.random.poisson(lam=4, size=10_000)

In [ ]:
npt.assert_equal(binned_spike_counts, binned_spike_counts.astype('uint8'))

AssertionError: 
Arrays are not equal

Mismatched elements: 10000 / 10000 (100%)
Max absolute difference among violations: 1
Max relative difference among violations: 1.
 ACTUAL: array([4, 3, 6, ..., 3, 4, 2], shape=(10000,), dtype=int32)
 DESIRED: array([5, 4, 7, ..., 4, 5, 3], shape=(10000,), dtype=uint8)

**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `np.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [ ]:
time_samples = np.arange(0, 10_000)

**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `np.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [ ]:
head_velocities = np.random.randint(low=-100000, high=100000, size=10_000)

**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data? Use `np.astype()` to make a transformed version of the data, then verify the data's integrity with `npt.assert_isclose()`.  

*Note*: These are floats, so you'll have to rather use  `npt.assert_isclose()` to verify that the transformed data has the same values as the original data, and note that here, you should specify an "absolute tolerance" (`atol`) and a "relative tolerance" (`rtol`), to say how different the new data is allowed to look from the old data.

In [41]:
time_seconds = np.arange(0, 200, .001)

In [45]:
npt.assert_allclose(time_seconds, time_seconds.astype(np.float32), atol=.001)

**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `np.astype()` to make a transformed version of the data, then verify the data's integrity with `npt.assert_isclose()`.

In [ ]:
firing_rates = np.random.exponential(scale=4, size=10_000)


---

## Section 3: Reducing Repetition with Dictionary Encoding


Many scientific datasets contain repeated categorical values:
  - Animal IDs
  - Experimental conditions
  - Trial labels
  - Brain region names
  - Behavioural states

Text takes up a lot of space; if we store these values as full strings (or large integers), we repeat the same information many times. 

Dictionary encoding is a common solution for this situation.  It reduces storage by splitting the categorical array into two arrays:

  1. An array of unique text values, storing each unique value only **once**.
  2. An array of integers, containing the index code of each category.

This changes the *representation* of the data without changing its meaning.

In this section, we will:

* Encode repeated values using `np.unique()`
* Compare memory usage before and after encoding
* Downcast integer codes to reduce size further
* Reconstruct the original data
* Compare with `pandas.factorize()`


### Reference

| Code                             | Description                                                 |
| -------------------------------- | ----------------------------------------------------------- |
| `np.unique(return_inverse=True)` | Return unique values and integer codes mapping each element |
| `array.nbytes`                   | Measure memory usage of an array                            |
| `astype()`                       | Convert codes to smaller integer dtype                      |
| `npt.assert_equal()`             | Verify exact equality after reconstruction                  |
| `pd.factorize()`                 | Encode repeated values into integer codes                   |
| `codes.max()`                    | Inspect largest encoded index                               |




### Exercises


**Exercise**: Take the a array of repeated animal IDs below and encode it using `uniques, codes = np.unique(return_inverse=True)`.  How much memory do the encoded representation and lookup table use together?


In [46]:
np.random.seed(42)
animal_ids = np.random.choice(["mouse", "rat", "human", "zebrafish"], size=1_000_000)
print(utils.format_bytes(animal_ids.nbytes))

36.00 MB


In [48]:
animal_ids.nbytes

36000000


**Exercise**: Use the encoded representation to reconstruct the original array (`reconstructed = uniques[codes]`), then:
  1. Verify that the reconstructed data matches the original. (`npt.assert_equal()`)
  2. Verify that the reconstructed data has the same size as the original.



In [53]:
categories, ids = np.unique(animal_ids, return_inverse=True)
ids

array([0, 3, 1, ..., 3, 3, 1], shape=(1000000,))

In [55]:
ids.astype(np.uint8).nbytes

1000000

In [ ]:
np.unique(animal_ids, )

array(['human', 'mouse', 'rat', 'zebrafish'], dtype='<U9')

In [56]:
categories[ids]

array(['human', 'zebrafish', 'mouse', ..., 'zebrafish', 'zebrafish',
       'mouse'], shape=(1000000,), dtype='<U9')


**Exercise**: Reducing Code Size Further with `astype()`.  Dictionary-encode the data below, then reduce the number of bits the `codes` array takes up, to further save space.  How small can we get the total transformed data?


In [249]:
np.random.seed(42)
brain_regions = np.random.choice(["visual cortex", "hippocampus", "thalamus", "auditory cortex"], size=3_000_000)
utils.format_bytes(brain_regions.nbytes)

'180.00 MB'

**Exercise**:  When saving this data, it is valuable to keep the arrays together in a single file; that way, the values aren't seperated from the codes.  This can be done by `np.savez()`!  

example: `np.savez('eeg.npz', time=times, voltages=volts)`

Save the dictionary-encoded brain regions you made in the previous exercise into a single file ".npz" file.

**Exercise**:  Okay, we can get even smaller.  Try out `np.savez_compressed()`, and compare the file size written to that when using `np.savez()`.

**Exercise**: Dictionary encoding is **not** helpful when the data is mostly made up of unique values.  Let's try it out!  

The code below generates an array of random DNA sequences.  In the cell below it, please use dictionary encoding on the data, and compare the sizes of the original dataset and the tranformed dataset.  Is there a difference?

In [57]:
import random
dna_seqs = np.array(["".join(random.choices("GCTA", k=60)) for _ in range(20_000)])
dna_seqs[:5]

array(['CTTTGACTCGAGAAGAGTCGCTGAGGTCTCAAACGTGTCGTGCAAATTTACCATTAAAAT',
       'ACACTGGGTTCCAAGAATGCTTCCCAATCGCGGTCGAGTGCGGAACATTGCATGCTACAA',
       'GAGCCAACGGGTCTTGGGCAATTATGAAGTCGGAGGAAATATATACTTATCCTAGAGCGT',
       'GACGACACGACGCAATAAGATGTGCCTATCAAAACTCCTGTCCGCGGCTTGCCCTATCCT',
       'GCCATATTCATTTACTGCATTTCGTCGCGTCCCCGTAACTTCCTTAGACAGTGGCAGGGA'],
      dtype='<U60')

---

## Section 4: Saving Fewer Zeros — Sparse Arrays

In many scientific datasets, most values are zero. Yes, actually zero.  Examples include:

* Spike trains binned at high temporal resolution (most of the time, no spikes are firing)
* Adjacency matrices in connectivity analyses (most things aren't heavily connected)
* Large masks or selection matrices (most data we're not tryng to select)

If we store these arrays "**densely**", we allocate memory for every element — including zeros. "**Sparse**" representations, on the other hand, store *only the non-zero values and their positions.*

Sparse arrays change representation without changing meaning. They can:

* Reduce memory usage dramatically
* Reduce disk storage
* Improve performance for certain operations

However, sparse arrays also introduce trade-offs:

* Not all NumPy operations are supported
* Converting between dense and sparse formats has a cost
* Sparse formats are beneficial only when many values are zero

In this section, we explore when sparse representations help and how to measure their impact.

### Reference

| Code                  | Description                             |
| --------------------- | --------------------------------------- |
| `scipy.sparse.csr_matrix()` | Create a compressed sparse row matrix   |
| `matrix.data`         | Non-zero values stored in sparse format |
| `matrix.indices`      | Column indices of non-zero values       |
| `matrix.indptr`       | Index pointer for row boundaries        |
| `matrix.toarray()`    | Convert sparse matrix back to dense     |
| `sparse.save_npz()`   | Save sparse matrix to compressed file   |
| `array.nbytes`        | Measure dense array memory usage        |
| `%time`               | Measure execution time in notebook      |
| `npt.assert_equal()`  | Verify equality after reconstruction    |

### Exercises

In [ ]:
# % pixi add --pypi scipy  

In [ ]:
from scipy import sparse

ModuleNotFoundError: No module named 'scipy'


**Exercise**: Creating a Sparse Matrix.  The large array below is roughly 99% zeros.

  1. Measure the size of the dense array.
  2. Convert it to a sparse representation wiht `sparse.csr_matrix()`
  3. Compare memory usage.   (note: you'll need to check size in three places: `data.data.nbytes`, `data.indices.nbytes`, and `data.indptr.nbytes`)
  4. Save the File to disk with `sparse.save_npz()`  How large is it on disk?

In [ ]:

n = 100_000
density = 0.01
dense = (np.random.rand(n) < density).astype(np.uint8)
print(utils.format_bytes(dense.nbytes))


**Exercise**: Read the Sparse Matrix, and convert it back to a Dense matrix with `data.toarray()`.  Verify that the data is the same as the original.